# Single-trial GRB localization

Run one Cosima simulation, reconstruct it with Revan, prepare a random one-second
background interval, and analyze the signal plus background with Mimrec.

Before running, install megalibpy in your notebook environment, prepare a separate
working MEGAlib installation, and replace the paths in **Define inputs and paths**.
The source file must reference valid geometry, orientation and spectrum resources;
Revan and Mimrec configurations must contain the intended scientific selections.
Set the background time bounds to a known usable interval in your TRA file.
Use a fresh workspace for each Cosima trial; running the entire notebook again
requires a new workspace. The native tools may take substantial time to run.

The analysis Python and MEGAlib Python environments may differ. The runtime
initializes native tools in child processes; ROOT image extraction also runs in a
separate compatible process. Do not import PyROOT into the analysis kernel.

The background window, direct TRA record counting, and CSV summary are local
research steps in this example, not a study scheduler or general pipeline API.
Background counts are measured after the background command's native selections;
other configured cuts still apply. The background time window is not carried into
the combined analysis. This example does not rescale exposure or align clocks.
A failed background extraction must be resolved before proceeding.

The localized coordinate is the maximum-intensity pixel center, not a centroid or
fitted source position. The summary excludes ARM, edge flags and peak intensity.
Saved execution outputs are omitted; run the cells in order after configuring inputs.


In [ ]:
# Standard library
import gzip
import random
import secrets
from pathlib import Path
import csv
import math

# Third-party libraries
from astropy import units as u
from astropy.coordinates import SkyCoord

# MEGAlib interfaces
import megalibpy
from megalibpy import (
    BaseParams,
    Cosima,
    MEGAlibRuntime,
    MEGAlibSource,
    Mimrec,
    Revan,
    RunConfig,
    SimulationWorkspace,
    Source,
    extract_image,
    run_command,
)

In [ ]:
def count_tra_events(file_path):
    """Count event records in a flat text .tra or .tra.gz file."""
    file_path = Path(file_path)
    opener = gzip.open if file_path.suffix == ".gz" else open

    with opener(file_path, "rb") as stream:
        return sum(1 for line in stream if line.strip() == b"SE")

# Define inputs and paths

In [ ]:
# this is the workspace path, grb-001 means a simulation space for a GRB
workspace_path = Path("localization-trials/grb-001")
working_space = SimulationWorkspace(workspace_path)

# Edit these paths for your MEGAlib installation and scientific inputs.
runtime = MEGAlibRuntime.from_setup_script("/path/to/megalib-setup.sh")
source_file = Path("/path/to/GRB.source")
config_dir = Path("/path/to/configurations")
revan_config = config_dir / "analysis.revan.cfg"
mimrec_config = config_dir / "analysis.mimrec.cfg"
geometry = "/path/to/geometry.geo.setup"

print("Workspace:", working_space.root)


# background components
# this should be updated when you use a different background piece
bkg_tra = Path("/path/to/background-day.tra")
bkg_tstart = 1835487300.0
bkg_tstop = bkg_tstart + 24*3600

# random 1s
duration = 1.0
t0 = random.uniform(bkg_tstart, bkg_tstop - duration)
t1 = t0 + duration
# bkg_path
bkg_1s_dir = working_space.root / "bkg"
bkg_1s_dir.mkdir(parents=True, exist_ok=True)
bkg_1s_path = bkg_1s_dir / "background_1s.tra.gz"

# seed
seed = secrets.randbelow(2**31 - 1) + 1

In [ ]:
# True coordinate

# read the source file
mega_source = MEGAlibSource(file_path = source_file)

# True position in the source file
true_coord = mega_source.source.coordinate

# Simulate GRB Signal

In [ ]:
# cosima simulation 
simulated = Cosima(runtime).simulate(
    source_file=source_file,
    workspace=working_space,
    seed=seed,
    log_mode="summary",
)

print("Simulation files:")
for path in simulated.outputs["sim"]:
    print(path)

print("\nSummary log:", simulated.stdout_path)
print("Stderr log:", simulated.stderr_path)
print("Manifest:", simulated.manifest_path)
print("Execution ID:", simulated.execution_id)

In [ ]:
if simulated.stdout_path is not None:
    print(simulated.stdout_path.read_text(errors="replace"))
else:
    print("No stdout was retained.")

capture = simulated.record["logs"]["capture"]["stdout"]
print("\nSummary found:", capture["summary_found"])

if capture["summary_missing"]:
    print(
        "The 'Summary for run' marker was not found in stdout. "
        "Please check the execution record and stderr."
    )

if simulated.stderr_path is not None:
    print("\n--- stderr ---")
    print(simulated.stderr_path.read_text(errors="replace"))

In [ ]:
# revan simualtion
revan = Revan(
    runtime,
    working_space=working_space,
    analysis_id="reconstruction",
    config_file=revan_config,
    geometry=geometry,
)


# event reconstruction
reconstructed_signal = revan.reconstruct()

print("Reconstructed files:")
for path in reconstructed_signal.outputs["tra"]:
    print(path)

print("\nStdout log:", reconstructed_signal.stdout_path)
print("Stderr log:", reconstructed_signal.stderr_path)
print("Execution ID:", reconstructed_signal.execution_id)

In [ ]:
# 定义 Mimrec 分析：绑定明确的上游、主配置和 geometry
mimrec_signal = Mimrec(
    runtime,
    working_space=working_space,
    analysis_id="grb_signal",
    input_data=reconstructed_signal,
    config_file=mimrec_config,
    geometry=geometry,
)


In [ ]:
# extract events

extracted_signal = mimrec_signal.extract_events(
    task="grb_signal",
)


print("Selected events:")
for path in extracted_signal.outputs["tra"]:
    print(path)

print("\nData group:", extracted_signal.output_dir)
print("Stdout log:", extracted_signal.stdout_path)
print("Stderr log:", extracted_signal.stderr_path)
print("Execution ID:", extracted_signal.execution_id)

In [ ]:
signal_counts = count_tra_events(extracted_signal.outputs["tra"][0])
signal_counts

# Make GRB 1s Background

In [ ]:
# Here we are running a simple mimrec command without creating a workspace

command = [
    "mimrec",
    "-g", str(geometry),
    "-f", str(bkg_tra),
    "-c", str(mimrec_config),
    "-o", str(bkg_1s_path),
    "-C", "EventSelections.TimeMode=1",
    "-C", f"EventSelections.Time.Min={t0:.9f}",
    "-C", f"EventSelections.Time.Max={t1:.9f}",
    "-n",
    "-x",
]

result = run_command(
    command,
    runtime=runtime,
    capture_output=True,
    check=False,
)

if result.returncode != 0:
    print(f"Mimrec failed with exit code {result.returncode}")
    print(result.stderr or result.stdout or "No diagnostic output was produced.")

In [ ]:
# get bkg counts

bkg_counts = count_tra_events(bkg_1s_path)
bkg_counts

# Combine the signal and background, run analysis

In [ ]:
# define the mimrec runtime for combined data

combined_analysis = Mimrec(
    runtime,
    working_space=working_space,
    analysis_id="grb_with_background",
    input_data=reconstructed_signal,
    config_file=mimrec_config,
    geometry=geometry,
    external_components=[bkg_1s_path],
)

In [ ]:
# extract the grb+bkg events
grb_with_background_extracted = combined_analysis.extract_events(
    task="grb_with_background_extracted",
)

In [ ]:
# get the total grb+bkg counts
grb_with_bkg_counts = count_tra_events(grb_with_background_extracted.outputs["tra"][0])
grb_with_bkg_counts

In [ ]:
# get spectrum
spectrum = combined_analysis.spectrum(
    input_data=grb_with_background_extracted,
    task="spectrum",
    output_name="spectrum.root",
)

In [ ]:
# get image
image = combined_analysis.image(
    input_data=grb_with_background_extracted,
    output_name="image.root",
)

In [ ]:
# get image png
image_png = combined_analysis.image(
    input_data=grb_with_background_extracted,
    output_name="image.png",
)

In [ ]:
# extract the root image and get localized center

image_data = extract_image(
    image,
    runtime=runtime,
)

localized_coord = image_data.peak_galactic_position  # Astropy SkyCoord

In [ ]:
# get angular separation
angular_error = true_coord.separation(localized_coord).deg
angular_error

# Localization trial summary

Write one row to `localization_summary.csv` in this trial's workspace. Rerunning
this cell replaces that trial's summary instead of appending duplicate rows.

- `SimNum` uses the workspace directory name; `Seed` comes from the actual Cosima execution record.
- Galactic coordinates and angular distance are in degrees.
- `S/N` is $S/\sqrt{S+B}$, using `Total_counts` for $S+B$; it is empty when total counts are zero.
- Background input and extraction timestamps identify the randomly selected interval, in native TRA time units (seconds).
- Peak intensity, edge flags, and ARM quantities are omitted.


In [ ]:
# Save one row for this localization trial.
summary_row = {
    "SimNum": working_space.root.name,
    "Seed": int(simulated.record["seed"]),
    "True_l": float(true_coord.galactic.l.deg),
    "true_b": float(true_coord.galactic.b.deg),
    "localized_l": float(localized_coord.galactic.l.deg),
    "localized_b": float(localized_coord.galactic.b.deg),
    "Angular_distance[deg]": float(angular_error),
    "Signal_counts": int(signal_counts),
    "Bkg_counts": int(bkg_counts),
    "Total_counts": int(grb_with_bkg_counts),
    "S/N": (
        float(signal_counts) / math.sqrt(grb_with_bkg_counts)
        if grb_with_bkg_counts > 0 else None
    ),
    "bkg_input_file": str(bkg_tra),
    "bkg_tstart": float(t0),
    "bkg_tstop": float(t1),
}

summary_path = working_space.root / "localization_summary.csv"
with summary_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(summary_row))
    writer.writeheader()
    writer.writerow(summary_row)

print(f"Summary saved to: {summary_path}")
summary_row
